# Topic 04. 헌혈 RFMTC 데이터 EDA

원본 과제의 핵심은 헌혈자의 Recency, Frequency, Monetary, Time, Class를 탐색하는 것입니다.
외부 데이터 호출 없이 실행되도록 작은 RFMTC 스타일 샘플 데이터를 직접 만들어 분석합니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. RFMTC 스타일 데이터 생성과 프로파일링

데이터 프로파일링은 행/열 수, 결측치, 요약통계, 클래스 비율을 먼저 확인하는 단계입니다.


In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2151050)
n = 80
donation = pd.DataFrame(
    {
        "Recency": rng.integers(1, 36, size=n),
        "Frequency": rng.poisson(lam=5, size=n) + 1,
        "Time": rng.integers(6, 80, size=n),
    }
)
donation["Monetary"] = donation["Frequency"] * 250
donation["Class"] = ((donation["Recency"] <= 12) & (donation["Frequency"] >= 5)).astype(int)

profile = {
    "shape": donation.shape,
    "missing_values": int(donation.isna().sum().sum()),
    "donation_rate": round(float(donation["Class"].mean()), 3),
}
profile


{'shape': (80, 5), 'missing_values': 0, 'donation_rate': 0.237}

## 2. 파생변수와 세그먼트 만들기

EDA에서는 원본 변수만 보는 것보다 분석 목적에 맞는 구간 변수를 만들면 해석이 쉬워집니다.
Recency와 Frequency를 기준으로 헌혈자 세그먼트를 나눕니다.


In [2]:
donation["RecencyGroup"] = pd.cut(
    donation["Recency"],
    bins=[0, 6, 12, 24, 36],
    labels=["very_recent", "recent", "dormant", "long_dormant"],
    include_lowest=True,
)
donation["FrequencyGroup"] = pd.cut(
    donation["Frequency"],
    bins=[0, 3, 6, donation["Frequency"].max() + 1],
    labels=["low", "medium", "high"],
    include_lowest=True,
)

segment_summary = (
    donation.groupby(["RecencyGroup", "FrequencyGroup"], observed=True)
    .agg(count=("Class", "size"), donation_rate=("Class", "mean"), avg_monetary=("Monetary", "mean"))
    .round(2)
    .reset_index()
)
segment_summary.head(10)


,RecencyGroup,FrequencyGroup,count,donation_rate,avg_monetary
0,very_recent,low,2,0.00,625.0
1,very_recent,medium,9,0.56,1250.0
2,very_recent,high,5,1.00,2000.0
3,recent,low,1,0.00,750.0
4,recent,medium,6,0.67,1250.0
5,recent,high,5,1.00,2150.0
6,dormant,low,2,0.00,750.0
7,dormant,medium,20,0.00,1250.0
8,dormant,high,8,0.00,2000.0
9,long_dormant,low,3,0.00,500.0


## 3. 상관관계와 가설 확인

가설 예시: 최근 헌혈자일수록 재헌혈 가능성이 높고, 헌혈 빈도가 높을수록 재헌혈 가능성이 높다.
상관계수는 인과관계가 아니라 함께 움직이는 정도를 보여줍니다.


In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

corr = donation[["Recency", "Frequency", "Monetary", "Time", "Class"]].corr().round(3)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, ax=ax)
ax.set_title("RFMTC Correlation")
plt.close(fig)

hypothesis_check = {
    "recency_vs_class_corr": float(corr.loc["Recency", "Class"]),
    "frequency_vs_class_corr": float(corr.loc["Frequency", "Class"]),
    "insight": "Recency는 낮을수록, Frequency는 높을수록 재헌혈 가능성이 커지는 구조로 생성되었습니다.",
}
hypothesis_check


{'recency_vs_class_corr': -0.604,
 'frequency_vs_class_corr': 0.328,
 'insight': 'Recency는 낮을수록, Frequency는 높을수록 재헌혈 가능성이 커지는 구조로 생성되었습니다.'}

## 정리

- EDA는 데이터 구조 확인, 결측치/통계 확인, 파생변수 생성, 가설 점검 순서로 진행합니다.
- RFMTC 변수는 고객 행동 분석과 비슷하게 헌혈자 유지 전략을 해석하는 데 쓸 수 있습니다.
- 상관관계는 의사결정의 단서이지만 인과관계로 단정하면 안 됩니다.
